# Baselines: naive fine-tune and full retrain

Trains both baselines over P1 to P4 on a Kaggle GPU and logs them to MLflow on DagsHub.
See `docs/design.md` sections 7 and 8, and `src/training/baselines.py`.

**Before you run**

1. Push the repository to GitHub. This notebook clones it, so unpushed commits are missing.
2. Settings: Accelerator **GPU T4 x2** (or P100), Internet **on**.
3. Add-ons, Secrets: add `DAGSHUB_TOKEN` (the same token as `.dvc/config.local`) and attach it
   to this notebook.

The token is only put into environment variables and is never printed. Estimated time is
well under an hour for both methods, not measured yet.

## 1. Credentials

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

# DVC reads its S3 keys from these variables, and MLflow reads the other three.
os.environ["AWS_ACCESS_KEY_ID"] = token
os.environ["AWS_SECRET_ACCESS_KEY"] = token
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/Akmal76/slangguard-data.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "Akmal76"
os.environ["MLFLOW_TRACKING_PASSWORD"] = token

## 2. Code and data

Kaggle already has torch and transformers. `mlflow-skinny` is the small MLflow client,
enough to log to a server. `dvc[s3]` is needed to pull the data.

In [ ]:
COMMIT = ""  # leave empty for the latest commit, or paste a commit SHA to pin the code

!git clone -q https://github.com/Akmal76/slangguard.git
%cd slangguard
if COMMIT:
    !git checkout -q {COMMIT}
!git log -1 --format="code at commit %h: %s"

In [ ]:
!pip install -q mlflow-skinny==3.18.0 'dvc[s3]'

In [ ]:
!dvc pull data/periods

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Turn on a GPU accelerator in the notebook settings.")
print(torch.cuda.get_device_name(0))

## 3. Train

One seed first (42) to check that everything works. Later, run the same cells with seeds
43 and 44 for the final numbers. Each run shows up in MLflow as `<method>-seed<seed>`.

In [ ]:
SEED = 42

In [ ]:
!python -m src.training.baselines --method naive --seed {SEED}

In [ ]:
!python -m src.training.baselines --method full --seed {SEED}

## 4. Results

Row t is the model after step t. Column i is the test set of period i. Cells above the
diagonal are periods the model has not seen yet.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

results = {
    m: json.loads(Path(f"outputs/{m}_seed{SEED}.json").read_text()) for m in ("naive", "full")
}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, (method, r) in zip(axes, results.items(), strict=True):
    image = ax.imshow(r["matrix"], vmin=0, vmax=1, cmap="Blues")
    for t, row in enumerate(r["matrix"]):
        for i, value in enumerate(row):
            ax.text(
                i,
                t,
                f"{value:.2f}",
                ha="center",
                va="center",
                color="white" if value > 0.6 else "black",
            )
    ax.set_xticks(range(4), [f"P{i + 1}" for i in range(4)])
    ax.set_yticks(range(4), [f"after P{t + 1}" for t in range(4)])
    ax.set_title(
        f"{method}: AvgF1 {r['avg_f1']:.3f}, forgetting {r['forgetting']:.3f}", fontsize=10
    )
fig.colorbar(image, ax=axes, shrink=0.8, label="macro-F1")
plt.show()

for method, r in results.items():
    print(f"{method}: training time {r['total_train_seconds'] / 60:.1f} min")